In [127]:
#数据探索与业务分析
import pandas as pd
import matplotlib.pyplot as plt

orders = pd.read_excel("../data/processed/orders_clean.xlsx")
orders.head()

,发货时间,订单编号（勿删）,商品ID（勿删）,单价,订单状态,退款状态,支付时间,商品名称,商品型号,销售批次,商品件数,邮费
0,2024-12-03,100001,7300251109,23.6,已完成,NaN,2024-12-17 12:10:02,徽章,无,徽章一批,1,6
1,2024-12-03,100002,7300251109,23.6,已完成,NaN,2024-12-04 11:39:25,徽章,无,徽章一批,1,6
2,2024-12-03,100003,7300251109,23.6,已完成,NaN,2024-12-04 00:29:16,徽章,无,徽章一批,1,6
3,2024-12-03,100004,7300251109,23.6,已完成,NaN,2024-12-07 17:30:34,徽章,无,徽章一批,1,6
4,2024-12-03,100005,7300251109,23.6,已完成,NaN,2024-12-03 11:49:04,徽章,无,徽章一批,1,6


In [128]:
# 核心指标
# GMV = 商品成交金额总和
orders["GMV"] = orders["单价"] * orders["商品件数"]

total_gmv = orders["GMV"].sum()

# 订单数
order_count = orders.shape[0]

# 销量
sales_volume = orders["商品件数"].sum()

# 平均客单价 = GMV ÷ 订单数
avg_order_value = total_gmv / order_count


summary = pd.DataFrame({
    "指标": ["GMV", "订单数", "销量", "平均客单价"],
    "数值": [
        total_gmv,
        order_count,
        sales_volume,
        avg_order_value
    ]
})

summary


,指标,数值
0,GMV,13561.400000
1,订单数,312.000000
2,销量,317.000000
3,平均客单价,43.466026


In [129]:
summary.to_excel("../data/processed/summary.xlsx",index=False)

## 项目概览分析结论

- 本项目共完成312笔订单，累计实现GMV 13561.4元，商品总销量达到317件。
- 平均客单价为43.47元，整体订单金额水平较为稳定。
- 从整体销售情况来看，项目收入主要由商品销量和不同商品的价格差异共同贡献，后续进一步分析不同商品及销售批次的销售表现。

In [130]:
product_analysis = orders.groupby("商品名称").agg(
    GMV=("GMV","sum"),
    销量=("商品件数","sum"),
    订单数=("订单编号（勿删）","count")
).reset_index()

product_analysis

,商品名称,GMV,销量,订单数
0,徽章,6466.4,274,269
1,棒球服,7095.0,43,43


In [131]:
product_analysis.to_excel("../data/processed/product_analysis.xlsx",index=False)

## 商品分析结论

- 徽章销量达到274件，占整体销量主要部分，是项目中的核心走量商品。
- 棒球服虽然销量仅43件，但GMV达到7095元，高于徽章，说明高客单价商品对收入贡献明显。
- 整体来看，徽章承担提升销量和订单规模的作用，棒球服承担提升销售额的作用，两类商品形成互补。

In [132]:
#销售批次
batch_order = [
    "徽章一批",
    "徽章二批",
    "棒球服首批",
    "徽章余量"
]

orders["销售批次"] = pd.Categorical(
    orders["销售批次"],
    categories=batch_order,
    ordered=True
)

In [133]:
batch_analysis = orders.groupby("销售批次").agg(
    GMV=("GMV", "sum"),
    销量=("商品件数", "sum"),
    订单数=("订单编号（勿删）", "count")
).reset_index()

batch_analysis["平均客单价"] = (
    batch_analysis["GMV"] / batch_analysis["订单数"]
)

batch_analysis

,销售批次,GMV,销量,订单数,平均客单价
0,徽章一批,2525.2,107,104,24.280769
1,徽章二批,3233.2,137,135,23.949630
2,棒球服首批,7095.0,43,43,165.000000
3,徽章余量,708.0,30,30,23.600000


In [134]:
batch_analysis.to_excel("../data/processed/batch_analysis.xlsx",index=False)


## 批次分析结论

- 棒球服首批GMV达到7095元，是各销售批次中最高的，但销量仅43件，主要原因是其平均客单价达到165元，属于高客单价产品。
- 徽章二批销售表现优于徽章一批，销量和GMV均有所提升，说明经过首批销售验证后，徽章产品仍保持较好的市场需求。
- 徽章余量批销售规模较小，GMV和销量均低于其他批次，符合后期剩余商品销售的特点。
- 整体来看，徽章属于高销量产品，承担稳定销售规模；棒球服属于高价值产品，对GMV贡献更明显。


In [135]:
#时间趋势分析
orders["支付时间"].min(), orders["支付时间"].max()

(Timestamp('2024-11-10 01:40:40'), Timestamp('2025-06-21 17:46:35'))

In [136]:
orders["日期"] = orders["支付时间"].dt.date

In [137]:
daily_sales = orders.groupby("日期").agg(
    GMV=("GMV", "sum"),
    销量=("商品件数", "sum"),
    订单数=("订单编号（勿删）", "count")
).reset_index()

daily_sales

,日期,GMV,销量,订单数
0,2024-11-10,1510.4,64,62
1,2024-11-11,259.6,11,10
2,2024-11-12,354.0,15,15
3,2024-11-14,23.6,1,1
4,2024-11-15,23.6,1,1
5,2024-11-16,23.6,1,1
6,2024-11-18,23.6,1,1
7,2024-11-21,23.6,1,1
8,2024-11-23,23.6,1,1
9,2024-11-25,23.6,1,1


In [138]:
daily_sales.to_excel("../data/processed/daily_sales.xlsx",index=False)

In [139]:
#GMV最高日期
daily_sales.sort_values("GMV",ascending=False).head()


,日期,GMV,销量,订单数
17,2024-12-10,2640.0,16,16
16,2024-12-09,2475.0,15,15
25,2025-01-05,1604.8,68,67
0,2024-11-10,1510.4,64,62
28,2025-01-08,778.8,33,32


In [140]:
#销量最高日期
daily_sales.sort_values("销量",ascending=False).head()


,日期,GMV,销量,订单数
25,2025-01-05,1604.8,68,67
0,2024-11-10,1510.4,64,62
28,2025-01-08,778.8,33,32
17,2024-12-10,2640.0,16,16
16,2024-12-09,2475.0,15,15


## 时间趋势分析结论

- 从每日销售表现来看，销售额和销量均存在明显波动，部分日期出现销售高峰。
- 2024-12-10和2024-12-09的GMV最高，分别达到2640元和2475元，说明这两个日期对整体收入贡献较大。
- 2025-01-05销量最高，达到68件，同时GMV达到1604.8元，说明该日期以较高销量贡献销售额。
- 不同日期的GMV与销量表现存在差异，可能与不同商品类型的销售特点有关，例如高客单价商品可能带来更高GMV，而低客单价商品可能贡献更多销量。

In [141]:
#棒球服尺码分析
jersey_orders = orders[orders["商品名称"] == "棒球服"]
jersey_orders.head()

,发货时间,订单编号（勿删）,商品ID（勿删）,单价,订单状态,退款状态,支付时间,商品名称,商品型号,销售批次,商品件数,邮费,GMV,日期
239,2025-01-17,100240,7318490693,165.0,已完成,NaN,2024-12-10 12:09:12,棒球服,L,棒球服首批,1,8,165.0,2024-12-10
240,2025-01-17,100241,7318490693,165.0,已完成,NaN,2024-12-10 23:33:15,棒球服,XL,棒球服首批,1,8,165.0,2024-12-10
241,2025-01-17,100242,7318490693,165.0,已完成,退款关闭,2024-12-13 23:25:14,棒球服,2XL,棒球服首批,1,8,165.0,2024-12-13
242,2025-01-17,100243,7318490693,165.0,已完成,NaN,2024-12-14 11:18:54,棒球服,M,棒球服首批,1,8,165.0,2024-12-14
243,2025-01-17,100244,7318490693,165.0,已完成,NaN,2024-12-14 22:06:12,棒球服,M,棒球服首批,1,8,165.0,2024-12-14


In [142]:
size_analysis = jersey_orders.groupby("商品型号").agg(
    销量=("商品件数", "sum"),
    订单数=("订单编号（勿删）", "count"),
    GMV=("GMV", "sum")
).reset_index()

size_analysis = size_analysis.sort_values(
    "销量",
    ascending=False
)

size_analysis

,商品型号,销量,订单数,GMV
3,M,14,14,2310.0
4,XL,12,12,1980.0
2,L,9,9,1485.0
0,2XL,5,5,825.0
1,3XL,3,3,495.0


In [143]:
size_analysis.to_excel("../data/processed/size_analysis.xlsx",index=False)

## 尺码分析结论

- 棒球服不同尺码之间存在一定销售差异，其中M码销量最高，达到14件，GMV为2310元，是主要销售尺码。
- XL码销量为12件，GMV为1980元，需求量仅次于M码，说明消费者购买需求主要集中在M和XL尺码。
- L码也保持了一定销售规模，而2XL和3XL等大尺码销量相对较低。
- 整体来看，棒球服尺码需求主要集中在中间尺码，后续备货时可根据历史销售数据优化各尺码生产比例。

In [144]:
#售后分析
refund_analysis = pd.DataFrame({
    "指标": [
        "总订单数",
        "退款申请订单数",
        "退款成功订单数",
        "实际退款率"
    ],
    "数值": [
        orders.shape[0],
        orders["退款状态"].notnull().sum(),
        (orders["退款状态"]=="退款完成(全额退款)").sum(),
        f"{(orders['退款状态']=='退款完成(全额退款)').mean():.2%}"
    ]
})

refund_analysis

,指标,数值
0,总订单数,312
1,退款申请订单数,3
2,退款成功订单数,1
3,实际退款率,0.32%


## 售后分析结论

- 本项目共包含312笔订单，其中有3笔订单发起退款申请，最终成功退款订单为1笔。
- 实际退款率为0.32%，整体退款规模较低，说明商品销售过程中的售后问题较少。
- 部分退款申请最终关闭，说明并非所有退款需求都会转化为实际退款。
- 后续可结合退款原因进一步分析用户售后需求，优化商品和服务流程。

# 项目总结

- 项目整体实现GMV 13561.4元，共产生312笔订单，销量317件。
- 商品方面，徽章承担主要销量贡献，棒球服虽然销量较低，但凭借较高客单价成为主要收入来源。
- 批次方面，徽章二批销售表现优于首批，说明产品经过初期验证后仍保持稳定需求。
- 时间分析发现销售存在明显高峰日期，不同商品类型对GMV和销量贡献存在差异。
- 尺码分析显示棒球服需求主要集中在M和XL尺码，可为后续备货提供参考。
- 售后方面，实际退款率较低，整体销售流程较稳定。